In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
from hotelling.spatial.osm import fetch_pois

# Fetch all Berlin supermarkets.  Cached to data/raw/OSM_POIs_Berlin_supermarket.parquet
# after the first run — subsequent calls are instant.
gdf = fetch_pois(type="supermarket", city="Berlin")
print(f"Supermarkets: {len(gdf)} rows, {gdf.shape[1]} columns")
gdf.head()

# Supermarkets gdf to grid crs
gdf = gdf.to_crs(grid.crs)

# Use only the centroids of the supermarket geometries
gdf.geometry = gdf.geometry.centroid

# Leave in gdf only the supermarkets that are within the grid
gdf = gdf[gdf.geometry.within(grid.union_all())]
print(f"Supermarkets within grid: {len(gdf)}")

In [ ]:
gdf['name'].sort_values().unique()
gdf[gdf['name'] == 'Lidl']['chain'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    # Plot these supermarkets
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    gdf.plot(ax=ax, color='firebrick', label='Supermarkets', alpha=0.7, markersize=1)
    grid.plot(ax=ax, facecolor='royalblue', edgecolor='none', label='Grid', alpha=0.2)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', label='Berlin Boundary', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)

    ax.set_axis_off()
    ax.legend()
    plt.title("Supermarkets in Berlin with Grid Overlay")
    plt.show()


In [ ]:
# Data cleaning and normalization
names = gdf.columns.tolist()
print("Columns in OSM supermarkets data:")
for name in names:
    print(f"- {name}")

# Fetch_pois already run a pre cleaning step of extracting the chain names. We check it again here to be sure.

# Candidates for holding supermarket chain names: 'brand', 'brand:wikidata', 'brand:wikipedia', 'name', 'shop', 'website', 'operator', 'contact:website', 'operator:wikidata', 'branch', 'email', 'brand:website', 'name:de', 'loc_name', 'owner', 'old_name', 'short_name', 'wikidata', 'wikipedia', 'alt_name', 'description', 'contact:email', 'identity', 'contact:facebook', 'membership', 'url', 'contact:instagram', 'contact:twitter', 'not:brand:wikidata'

check_gdf = gdf[['geometry', 'brand', 'brand:wikidata', 'brand:wikipedia', 'name', 'shop', 'website', 'operator', 'contact:website', 'operator:wikidata', 'branch', 'email', 'brand:website', 'name:de', 'loc_name', 'owner', 'old_name', 'short_name', 'wikidata', 'wikipedia', 'alt_name', 'description', 'contact:email', 'identity', 'contact:facebook', 'membership', 'url', 'contact:instagram', 'contact:twitter', 'not:brand:wikidata', 'chain']].copy()

# Drop all idx that have only geometry and no other info
check_gdf['has_info'] = check_gdf.drop(columns='geometry').notnull().any(axis=1)
check_gdf = check_gdf[check_gdf['has_info']]
print(f"Supermarkets with some info: {len(check_gdf)}")

In [ ]:
# Check out the rows with chain info
display(check_gdf[check_gdf['chain'].isna()][['name', 'brand', 'operator', 'website', 'chain']])

# Spotted: names: 'Denns Biomarkt' -> chain == 'Denns BioMarkt'
# 'Penny' -> Penny
# 'nahcity' -> REWE

check_gdf.loc[(check_gdf['name'] == 'Denns Biomarkt') & (check_gdf['chain'].isna()), 'chain'] = 'Denns BioMarkt'
check_gdf.loc[(check_gdf['name'] == 'Penny') & (check_gdf['chain'].isna()), 'chain'] = 'Penny'
check_gdf.loc[(check_gdf['name'] == 'nahcity') & (check_gdf['chain'].isna()), 'chain'] = 'REWE'

In [ ]:
check_gdf['chain'].value_counts()

In [ ]:
# Extracting unique chain names
check_gdf['chain'].sort_values().unique()

# In general, we have the following chains:
# - Edeka: Edeka, EDEKA
# - Netto: Netto Marken-Discount, Netto
# - Lidl: Lidl
# - REWE: REWE, Rewe
# - Aldi Nord: Aldi Nord
# - Penny: Penny
# - Denns BioMarkt: Denns BioMarkt
# - Bio Company: Bio Company
# - Alnatura: Alnatura
# - Kaufland: Kaufland
# - LPG BioMarkt: LPG BioMarkt
# - HIT: HIT
# - Norma: Norma

# The rest of the chains can be excluded as they are not part of the study or are too small to be relevant.

chain_mapping = {
    'EDEKA': ['Edeka', 'EDEKA'],
    'Netto': ['Netto Marken-Discount', 'Netto'],
    'Lidl': ['Lidl'],
    'REWE': ['REWE', 'Rewe'],
    'Aldi Nord': ['Aldi Nord'],
    'Penny': ['Penny'],
    'Denns BioMarkt': ['Denns BioMarkt'],
    'Bio Company': ['Bio Company'],
    'Alnatura': ['Alnatura'],
    'Kaufland': ['Kaufland'],
    'LPG BioMarkt': ['LPG BioMarkt'],
    'HIT': ['HIT'],
    'Norma': ['Norma']
}

# Create a new column 'chain_normalized' based on the mapping
def normalize_chain(chain):
    for normalized, variants in chain_mapping.items():
        if chain in variants:
            return normalized
    return None  # Return None for chains that are not in the mapping

check_gdf['chain_normalized'] = check_gdf['chain'].apply(normalize_chain)

chain_type = {
    'discount': ['Netto', 'Lidl', 'Aldi Nord', 'Penny', 'Norma'],
    'standard': ['Edeka', 'REWE', 'Kaufland', 'HIT'],
    'bio': ['Denns BioMarkt', 'Bio Company', 'Alnatura', 'LPG BioMarkt']
}

def classify_chain_type(chain):
    for ctype, chains in chain_type.items():
        if chain in chains:
            return ctype
    return None  # Return None for chains that are not classified

check_gdf['chain_type'] = check_gdf['chain_normalized'].apply(classify_chain_type)

In [ ]:
display(check_gdf['chain_normalized'].value_counts())

# Leave in check_gdf only the rows with normalized chain names
check_gdf = check_gdf[check_gdf['chain_normalized'].notnull()]
print(f"Supermarkets with normalized chain names: {len(check_gdf)}")

In [ ]:
if GENERATE_PLOTS:
    # Plot the supermarkets with normalized chain names
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    check_gdf.plot(ax=ax, column='chain_normalized', categorical=True, legend=True, markersize=5, alpha=0.7)
    grid.plot(ax=ax, facecolor='royalblue', edgecolor='none', label='Grid', alpha=0.2)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', label='Berlin Boundary', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    ax.legend()
    plt.title("Supermarkets in Berlin with Normalized Chain Names")
    plt.show()

In [ ]:
# Choose only important columns for the final gdf
final_gdf = check_gdf[['geometry', 'name', 'chain_normalized', 'chain_type']].copy()
final_gdf = final_gdf.rename(columns={'chain_normalized': 'chain'})

# Save the final cleaned and normalized supermarkets data to processed folder
final_gdf.to_parquet(PATH_PROCESSED / 'supermarkets.parquet', index=False)

In [ ]:
# The LCC anchor-store query has been moved into osm.py as the module-private
# constant `_LCC_TAGS`.  It encodes the following eight union blocks, each
# applied to nodes, ways, and relations inside the Berlin area:
#
#   shop=mall                              (shopping malls)
#   shop=department_store                  (Kaufhof/Galeria, Karstadt)
#   shop=chemist  + brand exists           (DM, Rossmann, Müller — chains only)
#   shop=variety_store                     (Action, Woolworth, Kik, Tedi …)
#   shop=electronics + brand exists        (MediaMarkt, Saturn, Conrad)
#   shop=doityourself                      (OBI, Hornbach, Bauhaus, Toom)
#   shop=furniture   + brand exists        (IKEA, XXXLutz, Mömax, Poco)
#   shop=sports      + brand exists        (Decathlon, Intersport, Sport Scheck)
#
# `fetch_pois(type="LCC")` builds the equivalent Overpass QL query at runtime
# using `_build_tag_filters` + `_build_overpass_query` and runs it against the
# Nominatim-resolved Berlin area ID — no manual area ID lookup needed.
from hotelling.spatial.osm import _LCC_TAGS  # inspect the tag list if needed
print(f"LCC tag blocks: {len(_LCC_TAGS)}")
for block in _LCC_TAGS:
    print(" ", block)

In [ ]:
# Fetch all LCC anchor stores for Berlin.
# Cached to data/raw/OSM_POIs_Berlin_LCC.parquet after the first run.
# No chain-name normalisation is applied for this type.
lcc_gdf = fetch_pois(type="LCC", city="Berlin")
print(f"LCC anchors: {len(lcc_gdf)} rows, {lcc_gdf.shape[1]} columns")
print(f"Columns: {list(lcc_gdf.columns)}")
lcc_gdf.head()

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    lcc_gdf.to_crs(berlin.crs).plot(ax=ax, markersize=5, legend=True, alpha=0.7)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    plt.show()

In [ ]:
# Compute the areas of the LCC
lcc_gdf['mall_area'] = lcc_gdf.to_crs(crs = 'EPSG:3035').geometry.area

In [ ]:
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    lcc_gdf[lcc_gdf['shop'] == 'mall'].to_crs(berlin.crs).plot(ax = ax, markersize=5, legend=True, alpha=0.7)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    plt.show()

In [ ]:
import numpy as np

# Join malls to grid cells. Ensure both layers share the same CRS before sjoin.
mall_gdf = lcc_gdf[(lcc_gdf['shop'] == 'mall') & (lcc_gdf['mall_area'] > 0)].copy()
mall_gdf = mall_gdf.to_crs(grid.crs)
mall_gdf = mall_gdf.reset_index(drop=True)

grid_malls = gpd.sjoin(grid, mall_gdf[['geometry', 'mall_area']], how='left', predicate='intersects')

# Compute the fraction of mall area that intersects with each grid cell
def compute_mall_intersection_fraction(row):
    if pd.isna(row['index_right']):
        return np.nan
    
    grid_geom = row.geometry
    mall_idx = int(row['index_right'])
    mall_geom = mall_gdf.iloc[mall_idx].geometry
    mall_area = row['mall_area']
    
    intersection_area = grid_geom.intersection(mall_geom).area
    fraction = intersection_area / mall_area if mall_area > 0 else 0
    return fraction

grid_malls['mall_intersection_fraction'] = grid_malls.apply(compute_mall_intersection_fraction, axis=1)
grid_malls['has_mall'] = grid_malls['mall_intersection_fraction'].notna()
grid_malls.head()


In [ ]:
grid_malls

In [ ]:
grid_only_malls = grid_malls[grid_malls['has_mall']]

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    grid_malls.to_crs(berlin.crs).plot(ax=ax, color = 'royalblue', legend=True, alpha=0.2)
    mall_gdf.to_crs(berlin.crs).plot(ax=ax, color='firebrick', label='Malls', alpha=1)
    grid_only_malls.to_crs(berlin.crs).plot(ax = ax, column='mall_area', cmap='OrRd', markersize=5, legend=True, alpha=0.7)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.5)
    ax.set_axis_off()
    plt.show()

In [ ]:
grid_malls.to_parquet(PATH_PROCESSED / 'grid_malls.parquet')
grid_malls

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_03_OSM.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")